# 03 · Distributed Training Patterns

**Goal:** see the two ways a cluster speeds up machine learning, and where Ray fits.

| Pattern | What is split across machines | Example in this notebook |
|---|---|---|
| **Many models** (task parallel) | Different *groups* of data, one model each | One model per age group, trained at the same time |
| **Parallel tuning** (task parallel) | Different *settings* of the same model | A 12-combination grid search, all combinations at once |
| **Data-parallel training** | One big model, each machine trains on a slice | Ray Train / TorchTrainer (classic ML cluster; code shown at the end) |

The first two patterns use Spark's **pandas UDFs**: you write a normal pandas + scikit-learn function, and Spark runs a
copy of it for every group or every setting, in parallel.

In [ ]:
%run ./Includes/_config

In [ ]:
import time
import pandas as pd

df = spark.table(TABLE)
GROUP_FEATURES = [c for c in FEATURES if c != "Age"]

## 1. Many models: one model per age group with `applyInPandas`
`groupBy("Age").applyInPandas(fn, schema)` sends each age group's rows to `fn` as a pandas DataFrame.
The 13 groups are trained at the same time on different workers. Each call returns one row of results.

In [ ]:
result_schema = "Age double, rows long, diabetes_rate double, test_auc double, train_seconds double"

def train_one_group(pdf: pd.DataFrame) -> pd.DataFrame:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split
    start = time.time()
    X, y = pdf[GROUP_FEATURES], pdf[LABEL]
    auc = float("nan")
    if y.nunique() == 2 and len(pdf) > 50:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
        model = LogisticRegression(max_iter=500).fit(X_tr, y_tr)
        auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    return pd.DataFrame([{"Age": pdf["Age"].iloc[0], "rows": len(pdf), "diabetes_rate": y.mean(),
                          "test_auc": auc, "train_seconds": time.time() - start}])

t0 = time.time()
per_group = df.groupBy("Age").applyInPandas(train_one_group, schema=result_schema).orderBy("Age").toPandas()
print(f"Trained {len(per_group)} models in {time.time() - t0:.1f} s (wall clock)")
per_group

**Read the table:** `train_seconds` is how long each model took. Because the groups trained in parallel, the wall-clock time
is far smaller than the sum of the column. This "many models" pattern is common in retail (one forecast per store) and
finance (one risk model per region).

## 2. Parallel hyperparameter search with `mapInPandas`
Each row of a small Spark DataFrame is one combination of settings. `mapInPandas` trains and scores a model for every row,
in parallel. The training sample travels to the workers together with the function.

In [ ]:
from itertools import product

sample_pdf = df.sample(fraction=0.3, seed=42).toPandas()
grid = pd.DataFrame([{"n_estimators": n, "max_depth": d, "min_samples_leaf": l}
                     for n, d, l in product([50, 100], [4, 8, 12], [1, 20])])
print(f"{len(grid)} combinations, {len(sample_pdf):,} training rows")

def score_settings(batches):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score
    X, y = sample_pdf[FEATURES], sample_pdf[LABEL]
    for batch in batches:
        rows = []
        for p in batch.to_dict("records"):
            start = time.time()
            model = RandomForestClassifier(n_estimators=int(p["n_estimators"]), max_depth=int(p["max_depth"]),
                                           min_samples_leaf=int(p["min_samples_leaf"]), n_jobs=1, random_state=42)
            auc = cross_val_score(model, X, y, cv=3, scoring="roc_auc").mean()
            rows.append({**p, "cv_auc": auc, "train_seconds": time.time() - start})
        yield pd.DataFrame(rows)

score_schema = "n_estimators long, max_depth long, min_samples_leaf long, cv_auc double, train_seconds double"
t0 = time.time()
results = (spark.createDataFrame(grid).repartition(len(grid))
           .mapInPandas(score_settings, schema=score_schema).toPandas()
           .sort_values("cv_auc", ascending=False))
wall = time.time() - t0
print(f"Wall clock {wall:.1f} s vs {results['train_seconds'].sum():.1f} s if run one after another")
results.head(12)

### Track the search in MLflow
One parent run holds the whole search; the results table is saved as an artifact.

In [ ]:
best = results.iloc[0]
with mlflow.start_run(run_name="parallel_rf_grid_search"):
    mlflow.log_params({f"best_{k}": int(best[k]) for k in ("n_estimators", "max_depth", "min_samples_leaf")})
    mlflow.log_metric("best_cv_auc", float(best["cv_auc"]))
    mlflow.log_metric("wall_clock_seconds", wall)
    mlflow.log_table(results, "grid_search_results.json")
print("Best settings:", best[["n_estimators", "max_depth", "min_samples_leaf"]].to_dict(), f"AUC {best['cv_auc']:.4f}")

## 3. Data-parallel training with Ray (classic ML cluster)
The patterns above train **many small models**. Deep learning often needs the opposite: **one large model** trained on all
the data. Ray Train runs the same PyTorch training loop on several workers, each reading its own slice, and combines what
they learn after every step.

```
             ┌── Worker 1: slice 1 ──┐
 Training ───┼── Worker 2: slice 2 ──┼──► gradients averaged ──► one shared model
  data       └── Worker 3: slice 3 ──┘
```

Ray needs real worker machines, so it runs on a **classic ML cluster** (in the full course: runtime 16.4 ML with 2+ workers),
not on serverless. The code in the full course looks like this:

```python
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer
import ray, ray.train, ray.train.torch

setup_ray_cluster(max_worker_nodes=2, num_cpus_worker_node=4)    # Ray starts on the Spark workers
ray.init()

def train_func(config):
    model = ray.train.torch.prepare_model(Net())                  # wraps the model for data-parallel training
    shard = ray.train.get_dataset_shard("train")                   # this worker's slice
    for epoch in range(config["epochs"]):
        for batch in shard.iter_torch_batches(batch_size=256):
            loss = train_step(model, batch)
        ray.train.report({"loss": float(loss)})

trainer = TorchTrainer(train_func, train_loop_config={"epochs": 5},
                       datasets={"train": ray.data.from_pandas(train_pdf)},
                       scaling_config=ScalingConfig(num_workers=2))
result = trainer.fit()
shutdown_ray_cluster()
```

## Key points
- **Many models** and **parallel tuning** split *tasks*: each worker trains its own small model (pandas UDFs).
- **Data-parallel training** splits *data* for one big model (Ray Train, Spark ML internally).
- Choose by the problem: thousands of small models → pandas UDFs; one big neural network → Ray.